# Kaleb Alejandro Aguilar Ávila
## Tarea 1 — Attention Recurrent Language Model and Machine Translation
**mLLMs / Procesamiento de Lenguaje Natural — CIMAT, Otoño 2026**  
**Profesor:** Dr. Adrián Pastor López Monroy

**LLM utilizado para Vibe Coding:** _(completar)_

---

### Índice

**0. Configuración**

**Parte 1 — Language Modelling (tuits MEX-A3T)**
- 1.0 Datos y vocabulario compartido
- 1.1 (Punto 1) Recurrent Language Model con Self-Attention
- 1.2 (Punto 2) Statistical Language Model
- 1.3 (Punto 3) Neural Language Model (Bengio)
- 1.4 (Punto 4) Comparación de perplejidades y ablación sin atención
- 1.5 (Punto 5) Generación de texto

**Parte 2 — Machine Translation inglés → español (Tatoeba)**
- 2.0 Datos
- 2.1 (Punto 1) Seq2Seq recurrente con atención Q, K, V
- 2.2 (Punto 2) Experimentos con X y 2X pares
- 2.3 (Punto 3) Evaluación en Test: BLEU, chrF y ejemplos

**3. Comentario general**

El código de los modelos vive en los módulos `.py` de `Task1_Language_Modelling/` y `Task2_Machine_Translation/`; este notebook sólo los importa y ejecuta los experimentos.

# 0. Configuración

In [ ]:
import os
import sys
import time
import random
import inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

T1_DIR = 'Task1_Language_Modelling'
T2_DIR = 'Task2_Machine_Translation'
sys.path.insert(0, os.path.abspath(T1_DIR))
sys.path.insert(0, os.path.abspath(T2_DIR))

# Parte 1
from common import (Vocab, load_tweets, train_dev_split, set_seed, get_device,
                    train_lm, perplexity)
import slm
import nlm
import rnn_attention_lm as rlm

# Parte 2
import mt_data as D
import seq2seq_attention as S

SEED = 1111
set_seed(SEED)
device = get_device()
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

os.makedirs('checkpoints', exist_ok=True)

---
# Parte 1 — Language Modelling

Datos: tuits MEX-A3T. **Train** (`mex20_train.txt`) para entrenar y **Val** (`mex20_val.txt`) para reportar la perplejidad.

| Punto | Modelo | Código |
|---|---|---|
| 1 | LSTM + Self-Attention (Q, K, V) | `Task1_Language_Modelling/rnn_attention_lm.py` |
| 2 | Statistical LM (n-gramas) | `Task1_Language_Modelling/slm.py` |
| 3 | Neural LM (Bengio) | `Task1_Language_Modelling/nlm.py` |
| 4 | Tabla de perplejidades + ablación sin atención | este notebook |
| 5 | Generación de texto | este notebook |

## 1.0 Datos y vocabulario compartido

Para que las perplejidades sean comparables, **todos los modelos usan el mismo tokenizador (`TweetTokenizer`), el mismo vocabulario y la misma convención**: cada tuit es independiente, se predicen todas sus palabras más `</s>`, y las palabras fuera de vocabulario se mapean a `<unk>`.

$$PPL = \exp\Big(-\frac{1}{N}\sum_i \log P(w_i \mid \text{historia}_i)\Big)$$

Train se divide 90/10 en *train* / *dev*; *dev* sólo se usa para early stopping. **Val se usa únicamente para reportar la perplejidad final.**

In [ ]:
lm_train_full = load_tweets(os.path.join(T1_DIR, 'data', 'mex20_train.txt'))
lm_val = load_tweets(os.path.join(T1_DIR, 'data', 'mex20_val.txt'))
lm_train, lm_dev = train_dev_split(lm_train_full, dev_frac=0.1, seed=SEED)
print(f'train_full={len(lm_train_full)} | train={len(lm_train)} | dev={len(lm_dev)} | val={len(lm_val)}')

VOCAB_SIZE = 5000
vocab = Vocab(max_size=VOCAB_SIZE, min_freq=1, lower=True).fit(lm_train_full)
print('|V| =', len(vocab))
print(f'OOV rate val: {vocab.oov_rate(lm_val):.2%}')
lm_train_full[:5]

## 1.1 (Punto 1) Recurrent Language Model con Self-Attention

**Arquitectura** (`rnn_attention_lm.py`):
- Embeddings (256) → LSTM apilada de 2 capas (512, **unidireccional**, para no ver el futuro).
- **Self-attention causal multi-cabeza (4) tipo Transformer** sobre los estados ocultos `h`:
  - `Q = h W_Q`, `K = h W_K`, `V = h W_V`
  - `A = softmax(Q Kᵀ / √d_k + máscara causal)`, `c_t = Σ_{j≤t} A_tj V_j`
- `z_t = tanh(W_c [h_t; c_t])` → dropout → softmax sobre el vocabulario.

**Entrenamiento:** Adam (lr = 1e-3), gradient clipping = 1.0, dropout = 0.4, `ReduceLROnPlateau`, early stopping sobre *dev*.

In [ ]:
print(inspect.getsource(rlm.CausalSelfAttention))

In [ ]:
lm_results = []

RNN_CFG = dict(emb_dim=256, hidden_dim=512, num_layers=2, n_heads=4, dropout=0.4, rnn_type='lstm')
RNN_TRAIN_CFG = dict(lr=1e-3, num_epochs=30, patience=3, clip=1.0)

rnn_train_loader = rlm.make_loader(lm_train, vocab, batch_size=64, shuffle=True)
rnn_dev_loader   = rlm.make_loader(lm_dev,   vocab, batch_size=128)
rnn_val_loader   = rlm.make_loader(lm_val,   vocab, batch_size=128)

set_seed(SEED)
rnn_att = rlm.RNNAttentionLM(len(vocab), use_attention=True, pad_id=vocab.pad_id, **RNN_CFG)
print(rnn_att)
print('Parámetros:', rlm.count_parameters(rnn_att))

t0 = time.time()
hist_att = train_lm(rnn_att, rnn_train_loader, rnn_dev_loader, device, vocab.pad_id,
                    save_path='checkpoints/rnn_attention.pt', **RNN_TRAIN_CFG)
print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

ppl = perplexity(rnn_att, rnn_val_loader, device, vocab.pad_id)
print(f'LSTM + Self-Attention -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': 'LSTM + Self-Attention (QKV)', 'PPL val': ppl})

**Visualización de los pesos de atención**

In [ ]:
toks, A = rlm.attention_map(rnn_att, vocab, 'la verdad es que no me importa lo que digan', device)
plt.figure(figsize=(6, 5))
plt.imshow(A.numpy(), cmap='viridis')
plt.xticks(range(len(toks)), toks, rotation=90); plt.yticks(range(len(toks)), toks)
plt.xlabel('Key'); plt.ylabel('Query'); plt.colorbar(); plt.title('Self-attention (promedio de cabezas)')
plt.show()

**Observaciones (Punto 1):** _(completar: hiperparámetros elegidos y por qué, dificultades, alucinaciones del LLM, etc.)_

## 1.2 (Punto 2) Statistical Language Model

Trigramas (`slm.py`) con:
- **Laplace / add-k:** $P(w\mid ctx) = \frac{C(ctx, w) + k}{C(ctx) + k|V|}$
- **Interpolación lineal (Jelinek-Mercer):** $P = \lambda_3 P_{ML}(w\mid w_{i-2}, w_{i-1}) + \lambda_2 P_{ML}(w\mid w_{i-1}) + \lambda_1 P_{add\text{-}k}(w)$

In [ ]:
slm_laplace = slm.NgramLM(vocab, n=3, smoothing='laplace', k=1.0).fit(lm_train_full)
ppl = slm_laplace.perplexity(lm_val)
print(f'SLM trigramas Laplace      -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': 'SLM trigramas (Laplace)', 'PPL val': ppl})

slm_interp = slm.NgramLM(vocab, n=3, smoothing='interpolation', k=0.1,
                         lambdas=[0.1, 0.3, 0.6]).fit(lm_train_full)
ppl = slm_interp.perplexity(lm_val)
print(f'SLM trigramas interpolado  -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': 'SLM trigramas (interpolación)', 'PPL val': ppl})

Ajuste de las lambdas de interpolación usando **dev** (nunca Val):

In [ ]:
best = None
for lambdas in [[0.1, 0.3, 0.6], [0.2, 0.4, 0.4], [0.3, 0.4, 0.3], [0.2, 0.5, 0.3], [0.4, 0.4, 0.2]]:
    m = slm.NgramLM(vocab, n=3, smoothing='interpolation', k=0.1, lambdas=lambdas).fit(lm_train)
    p = m.perplexity(lm_dev)
    print(lambdas, f'dev PPL = {p:.2f}')
    if best is None or p < best[0]:
        best = (p, lambdas)
print('Mejores lambdas:', best[1])

slm_best = slm.NgramLM(vocab, n=3, smoothing='interpolation', k=0.1, lambdas=best[1]).fit(lm_train_full)
ppl = slm_best.perplexity(lm_val)
print(f'SLM interpolado (lambdas ajustadas en dev) -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': f'SLM trigramas (interp. {best[1]})', 'PPL val': ppl})

**Observaciones (Punto 2):** _(completar)_

## 1.3 (Punto 3) Neural Language Model de Bengio

Ventana de 3 palabras → embeddings concatenados (100) → `tanh` (256) → dropout → softmax (`nlm.py`, basado en la Tarea 5 de NLP).

In [ ]:
N = 4  # 3 palabras de contexto
nlm_train_loader = nlm.make_loader(lm_train, vocab, N, batch_size=256, shuffle=True)
nlm_dev_loader   = nlm.make_loader(lm_dev,   vocab, N, batch_size=512)
nlm_val_loader   = nlm.make_loader(lm_val,   vocab, N, batch_size=512)

set_seed(SEED)
bengio = nlm.BengioLM(len(vocab), N=N, emb_dim=100, hidden_dim=256, dropout=0.3, pad_id=vocab.pad_id)
t0 = time.time()
hist_nlm = train_lm(bengio, nlm_train_loader, nlm_dev_loader, device, vocab.pad_id,
                    lr=1e-3, num_epochs=30, patience=3, save_path='checkpoints/bengio.pt')
print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

ppl = perplexity(bengio, nlm_val_loader, device, vocab.pad_id)
print(f'NLM Bengio -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': f'NLM Bengio (N={N})', 'PPL val': ppl})

**Observaciones (Punto 3):** _(completar)_

## 1.4 (Punto 4) Comparación de perplejidades

### Ablación: mismo modelo del Punto 1 **sin atención**
`use_attention=False`: se elimina el bloque Q, K, V y `z_t = tanh(W_c h_t)`; todo lo demás (datos, vocabulario, hiperparámetros, semilla) es idéntico.

In [ ]:
set_seed(SEED)
rnn_noatt = rlm.RNNAttentionLM(len(vocab), use_attention=False, pad_id=vocab.pad_id, **RNN_CFG)
print('Parámetros:', rlm.count_parameters(rnn_noatt))

t0 = time.time()
hist_noatt = train_lm(rnn_noatt, rnn_train_loader, rnn_dev_loader, device, vocab.pad_id,
                      save_path='checkpoints/rnn_no_attention.pt', **RNN_TRAIN_CFG)
print(f'Tiempo de entrenamiento: {time.time() - t0:.1f}s')

ppl = perplexity(rnn_noatt, rnn_val_loader, device, vocab.pad_id)
print(f'LSTM sin atención -> PPL val = {ppl:.2f}')
lm_results.append({'Modelo': 'LSTM sin atención (ablación)', 'PPL val': ppl})

In [ ]:
plt.figure(figsize=(7, 4))
for name, h in [('Bengio', hist_nlm), ('LSTM + Att', hist_att), ('LSTM', hist_noatt)]:
    plt.plot(range(1, len(h['dev_ppl']) + 1), h['dev_ppl'], marker='o', label=name)
plt.xlabel('Época'); plt.ylabel('PPL dev'); plt.title('Curvas de validación (dev)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Tabla de resultados (perplejidad sobre Val)

In [ ]:
lm_table = pd.DataFrame(lm_results).sort_values('PPL val').reset_index(drop=True)
lm_table['PPL val'] = lm_table['PPL val'].round(2)
lm_table

**Observaciones (Punto 4):** _(completar: ¿se derrotó al SLM y al NLM? ¿qué efecto tuvo quitar la atención?)_

## 1.5 (Punto 5) Generación de texto — 2 ejemplos por modelo

In [ ]:
prefixes = ['yo creo que', 'la verdad es']
TEMP = 0.8
set_seed(SEED)

for prefix in prefixes:
    print('=' * 80)
    print(f'Prefijo: "{prefix}"')
    print(' SLM         :', slm_best.generate(prefix, max_len=25, temperature=TEMP, seed=SEED))
    print(' NLM Bengio  :', nlm.generate(bengio, vocab, prefix, max_len=25, temperature=TEMP, device=device))
    print(' LSTM + Att  :', rlm.generate(rnn_att, vocab, prefix, max_len=25, temperature=TEMP, device=device))
    print(' LSTM sin Att:', rlm.generate(rnn_noatt, vocab, prefix, max_len=25, temperature=TEMP, device=device))

**Demo: log-likelihood de oraciones** (una oración bien formada vs. una permutación)

In [ ]:
oraciones = ['yo creo que sí', 'sí que creo yo', 'no manches me cae mal']
rows = []
for s in oraciones:
    rows.append({'oración': s,
                 'SLM': slm_best.sentence_logprob(s),
                 'NLM': nlm.sentence_logprob(bengio, vocab, s, device),
                 'LSTM + Att': rlm.sentence_logprob(rnn_att, vocab, s, device),
                 'LSTM': rlm.sentence_logprob(rnn_noatt, vocab, s, device)})
pd.DataFrame(rows).round(2)

**Observaciones (Punto 5) y comentario de la Parte 1:** _(completar)_

---
# Parte 2 — Machine Translation (inglés → español)

Datos: Tatoeba (`tatoeba_train.tsv` 80 %, `tatoeba_val.tsv` 10 %, `tatoeba_test.tsv` 10 %).
- **Train** → entrenamiento (subconjuntos anidados de X y 2X pares).
- **Val** → early stopping / decisiones de hiperparámetros.
- **Test** → sólo evaluación final (BLEU y chrF). **Nunca** se usa para tomar decisiones.

Código: `Task2_Machine_Translation/mt_data.py` (datos, tokenización, vocabularios) y `Task2_Machine_Translation/seq2seq_attention.py` (modelo, entrenamiento, traducción y métricas).

## 2.0 Datos

In [ ]:
train_pairs = D.load_tatoeba(os.path.join(T2_DIR, 'data', 'tatoeba_train.tsv'))
val_pairs   = D.load_tatoeba(os.path.join(T2_DIR, 'data', 'tatoeba_val.tsv'))
test_pairs  = D.load_tatoeba(os.path.join(T2_DIR, 'data', 'tatoeba_test.tsv'))
print(f'|Train|={len(train_pairs)} | |Val|={len(val_pairs)} | |Test|={len(test_pairs)}')

lens_en = [len(D.tokenize(en)) for en, _ in train_pairs]
lens_es = [len(D.tokenize(es)) for _, es in train_pairs]
print(f'Longitud media (tokens) en={np.mean(lens_en):.1f} es={np.mean(lens_es):.1f} | '
      f'p99 en={np.percentile(lens_en, 99):.0f} es={np.percentile(lens_es, 99):.0f}')
train_pairs[:5]

## 2.1 (Punto 1) Seq2Seq recurrente con atención Q, K, V

**No es un Transformer:** encoder y decoder son recurrentes (GRU); sólo el bloque de atención es tipo Transformer.

- **Encoder:** Embedding (256) → GRU bidireccional de 2 capas (512) → salidas `H` (B, S, 1024).
- **Decoder:** GRU de 2 capas (512) con *input feeding* `[emb(y_{t-1}); z_{t-1}]`. En cada paso `t`:
  - **Query:** `Q = s_t · W_Q` ← estado oculto del decoder
  - **Keys:** `K = H · W_K` ← salidas del encoder
  - **Values:** `V = H · W_V` ← salidas del encoder
  - **Pesos de atención:** `α_t = softmax(Q Kᵀ / √d_k)` con máscara sobre `<pad>` (4 cabezas)
  - **Contexto:** `c_t = α_t · V` → `z_t = tanh(W_c [s_t; c_t])` → `softmax(W_o z_t)`
- **Entrenamiento:** teacher forcing, Adam (lr = 1e-3), label smoothing 0.1, gradient clipping 1.0, `ReduceLROnPlateau`, early stopping sobre la pérdida de Val. Decodificación greedy.

In [ ]:
print(inspect.getsource(S.QKVAttention))

## 2.2 (Punto 2) Experimentos con X y 2X pares

Ambos experimentos usan **exactamente la misma arquitectura y configuración de entrenamiento**; sólo cambia el número de pares de Train. Los X pares del experimento 1 son un **subconjunto** de los 2X del experimento 2 (Train se baraja una sola vez con semilla fija; se toman los primeros 2X y, de ésos, los primeros X).

Ajusta `X` según tus recursos de cómputo (debe cumplirse 2X ≤ |Train|). Los vocabularios se construyen sólo con los pares de entrenamiento de cada experimento.

In [ ]:
X = 50_000   # <- ajustar según recursos (2X = 100,000 <= |Train|)

MT_DATA_CFG  = dict(lower=True, min_freq=2, max_len_train=30, batch_size=128)
MT_MODEL_CFG = dict(emb_dim=256, hidden_dim=512, num_layers=2, attn_dim=256, n_heads=4, dropout=0.3)
MT_TRAIN_CFG = dict(lr=1e-3, num_epochs=20, patience=3, clip=1.0, teacher_forcing=1.0, label_smoothing=0.1)

subset_x, subset_2x = D.make_nested_subsets(train_pairs, X, seed=SEED)
assert set(subset_x) <= set(subset_2x)
print(f'Experimento 1: {len(subset_x)} pares | Experimento 2: {len(subset_2x)} pares')

In [ ]:
def run_mt_experiment(name, pairs):
    """Construye vocabularios con `pairs`, entrena con early stopping sobre Val y regresa todo lo necesario."""
    set_seed(SEED)
    src_vocab, tgt_vocab = D.build_vocabs(pairs, min_freq=MT_DATA_CFG['min_freq'], lower=MT_DATA_CFG['lower'])
    print(f'[{name}] |V_en|={len(src_vocab)} |V_es|={len(tgt_vocab)}')

    train_loader = D.make_loader(pairs, src_vocab, tgt_vocab, MT_DATA_CFG['batch_size'], shuffle=True,
                                 lower=MT_DATA_CFG['lower'], max_len=MT_DATA_CFG['max_len_train'])
    val_loader = D.make_loader(val_pairs, src_vocab, tgt_vocab, MT_DATA_CFG['batch_size'],
                               lower=MT_DATA_CFG['lower'])

    model = S.Seq2SeqAttention(len(src_vocab), len(tgt_vocab), src_pad_id=src_vocab.pad_id,
                               tgt_pad_id=tgt_vocab.pad_id, **MT_MODEL_CFG).to(device)
    print(f'[{name}] parámetros: {S.count_parameters(model):,}')

    # BLEU sobre una muestra fija de Val para monitorear (NO Test)
    val_sample = val_pairs[:1000]
    def val_bleu():
        hyps = S.translate(model, [en for en, _ in val_sample], src_vocab, tgt_vocab, device,
                           lower=MT_DATA_CFG['lower'])
        return S.compute_metrics(hyps, [es for _, es in val_sample], lowercase=MT_DATA_CFG['lower'])['BLEU']

    t0 = time.time()
    history = S.fit(model, train_loader, val_loader, tgt_vocab.pad_id, device,
                    save_path=f'checkpoints/seq2seq_{name}.pt', val_bleu_fn=val_bleu, **MT_TRAIN_CFG)
    train_time = time.time() - t0
    print(f'[{name}] tiempo de entrenamiento: {train_time / 60:.1f} min')
    return dict(model=model, src_vocab=src_vocab, tgt_vocab=tgt_vocab, history=history, train_time=train_time)

### Experimento 1 — X pares

In [ ]:
exp_x = run_mt_experiment('X', subset_x)

### Experimento 2 — 2X pares

In [ ]:
exp_2x = run_mt_experiment('2X', subset_2x)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name, exp in [('X', exp_x), ('2X', exp_2x)]:
    h = exp['history']
    ep = range(1, len(h['val_loss']) + 1)
    ax[0].plot(ep, h['train_loss'], '--', label=f'train {name}')
    ax[0].plot(ep, h['val_loss'], marker='o', label=f'val {name}')
    ax[1].plot(ep, h['val_bleu'], marker='o', label=name)
ax[0].set_title('Loss'); ax[0].set_xlabel('Época'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title('BLEU (muestra de Val)'); ax[1].set_xlabel('Época'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

**Observaciones (Punto 2):** _(completar: valor de X elegido y por qué, recursos de cómputo, tiempo de entrenamiento)_

## 2.3 (Punto 3) Evaluación en Test — BLEU y chrF

In [ ]:
test_src = [en for en, _ in test_pairs]
test_ref = [es for _, es in test_pairs]

mt_rows = []
for name, exp, n in [('X', exp_x, X), ('2X', exp_2x, 2 * X)]:
    t0 = time.time()
    exp['test_hyps'] = S.translate(exp['model'], test_src, exp['src_vocab'], exp['tgt_vocab'], device,
                                   lower=MT_DATA_CFG['lower'])
    m = S.compute_metrics(exp['test_hyps'], test_ref, lowercase=MT_DATA_CFG['lower'])
    print(name, m['bleu_obj'], '|', m['chrf_obj'], f'| {time.time() - t0:.0f}s')
    mt_rows.append({'Experimento': name, 'Pares de Train': n, 'Épocas': len(exp['history']['val_loss']),
                    'Val loss': round(exp['history']['best_val_loss'], 3),
                    'BLEU (Test)': round(m['BLEU'], 2), 'chrF (Test)': round(m['chrF'], 2),
                    'Tiempo (min)': round(exp['train_time'] / 60, 1)})

pd.DataFrame(mt_rows)

### Ejemplos de Test (≥ 10)

In [ ]:
rng = random.Random(SEED)
idx = rng.sample(range(len(test_pairs)), 12)
examples = pd.DataFrame({
    'Inglés (original)': [test_src[i] for i in idx],
    'Referencia (es)': [test_ref[i] for i in idx],
    'Modelo X': [exp_x['test_hyps'][i] for i in idx],
    'Modelo 2X': [exp_2x['test_hyps'][i] for i in idx],
})
with pd.option_context('display.max_colwidth', None):
    display(examples)

### Mapa de atención (ejemplo)

In [ ]:
sentence = test_src[idx[0]]
src_toks, out_toks, A = S.translate_with_attention(exp_2x['model'], sentence, exp_2x['src_vocab'],
                                                   exp_2x['tgt_vocab'], device)
plt.figure(figsize=(6, 5))
plt.imshow(A.numpy(), cmap='viridis', aspect='auto')
plt.xticks(range(len(src_toks)), src_toks, rotation=90); plt.yticks(range(len(out_toks)), out_toks)
plt.xlabel('Inglés (Keys/Values)'); plt.ylabel('Español generado (Queries)'); plt.colorbar()
plt.title('Pesos de atención (promedio de cabezas) — modelo 2X'); plt.show()

**Observaciones (Punto 3):** _(completar: comparación X vs 2X, errores y diferencias observadas en los ejemplos)_

---
# 3. Comentario general

_(completar: LLM usado para Vibe Coding, decisiones tomadas, dificultades, alucinaciones, recursos de cómputo y tiempos de entrenamiento)_